# TS-GNN: Temporal Sheaf Graph Neural Network
**Allele-conditioned GRN rewiring via temporal sheaf diffusion**

Questo notebook esegue l'intera pipeline TS-GNN su Google Colab con GPU gratuita.

### Setup
1. **Runtime > Change runtime type > T4 GPU**
2. Esegui tutte le celle in ordine

## 0. Verifica GPU

In [3]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA disponibile: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    
else:
    print("ATTENZIONE: Nessuna GPU. Vai su Runtime > Change runtime type > T4 GPU")

PyTorch: 2.10.0+cu128
CUDA disponibile: True
GPU: Tesla T4


## 1. Upload del progetto e installazione dipendenze

In [4]:
# OPZIONE A: Upload da Google Drive (consigliato)
# Carica la cartella ts-gnn nel tuo Google Drive, poi:
from google.colab import drive
drive.mount('/content/drive')

# Modifica questo path se la cartella e' in una posizione diversa
PROJECT_DIR = '/content/drive/MyDrive/ts-gnn'

import os
if not os.path.exists(PROJECT_DIR):
    print(f"ERRORE: {PROJECT_DIR} non trovato!")
    print("Opzioni:")
    print("  1. Carica la cartella ts-gnn su Google Drive")
    print("  2. Modifica PROJECT_DIR con il path corretto")
    print("  3. Usa l'Opzione B nella cella successiva (upload zip)")
else:
    print(f"Progetto trovato: {PROJECT_DIR}")
    # Copia in locale per velocita'
    !cp -r {PROJECT_DIR} /content/ts-gnn
    PROJECT_DIR = '/content/ts-gnn'
    print(f"Copiato in {PROJECT_DIR}")

Mounted at /content/drive
ERRORE: /content/drive/MyDrive/ts-gnn non trovato!
Opzioni:
  1. Carica la cartella ts-gnn su Google Drive
  2. Modifica PROJECT_DIR con il path corretto
  3. Usa l'Opzione B nella cella successiva (upload zip)


In [5]:
# OPZIONE B: Upload diretto (alternativa se non usi Drive)
# Decommentare queste righe e commentare la cella sopra

# from google.colab import files
# print("Carica il file ts-gnn.zip")
# uploaded = files.upload()  # Carica ts-gnn.zip
# !unzip -q ts-gnn.zip -d /content/
# PROJECT_DIR = '/content/ts-gnn'

In [6]:
# Installa dipendenze
!pip install -q torch-geometric torch-sparse torch-scatter \
    scanpy scvelo anndata harmonypy scrublet \
    decoupler wandb hydra-core omegaconf \
    networkx plotly seaborn pyyaml \
    fair-esm 2>&1 | tail -5

print("\nDipendenze installate!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.1/60.1 kB 7.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 295.7/295.7 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 85.0/85.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.2/9.2 MB 128.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 kB 4.2 MB/s eta 0:00:00

Dipendenze installate!


In [7]:
# Aggiungi il progetto al Python path
import sys
sys.path.insert(0, f'{PROJECT_DIR}/src')

# Verifica import
from tsgnn.model.tsgnn import TSGNN
from tsgnn.model.sheaf import SheafDiffusionLayer
from tsgnn.training.loss import TSGNNLoss
from tsgnn.training.trainer import TSGNNTrainer
print("Tutti i moduli importati correttamente!")

ModuleNotFoundError: No module named 'tsgnn'

## 2. Configurazione

In [ ]:
import yaml

# Carica config di default
with open(f'{PROJECT_DIR}/configs/default.yaml') as f:
    config = yaml.safe_load(f)

# Override per Colab (piu' veloce per test)
config['training'] = config.get('training', {})
config['training']['lr'] = 1e-3
config['training']['max_epochs'] = 100       # Riduci per test veloce
config['training']['patience'] = 15
config['training']['max_grad_norm'] = 1.0
config['training']['mixed_precision'] = True  # fp16 su GPU
config['training']['use_wandb'] = False       # Metti True se vuoi logging

config['loss'] = config.get('loss', {})
config['loss']['lambda_1'] = 0.1
config['loss']['lambda_2'] = 0.5
config['loss']['lambda_3'] = 0.01
config['loss']['tau'] = 0.5

# Parametri modello
N = 500          # Numero geni
E = 2000         # Numero edge nel GRN
d = 4            # Stalk dimension
K = 10           # Temporal bins
ESM_DIM = 1280   # Dimensione embedding ESM-2

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Config pronta. Device: {DEVICE}")
print(f"Modello: N={N}, E={E}, d={d}, K={K}")

## 3. Generazione ESM-2 Embeddings

In [ ]:
from tsgnn.data.allele import (
    TP53_HOTSPOT_MUTATIONS, TP53_CANONICAL_SEQUENCE,
    create_mutant_sequence
)

ALLELES = ['WT', 'R175H', 'R273H', 'R248W', 'R282W', 'G245S', 'Y220C']

# Prova a generare embeddings reali con ESM-2
try:
    import esm
    print("Caricamento ESM-2 (esm2_t33_650M_UR50D)...")
    esm_model, alphabet = esm.pretrained.esm2_t33_650M_UR50D()
    esm_model = esm_model.to(DEVICE).eval()
    batch_converter = alphabet.get_batch_converter()

    # Genera sequenze mutanti
    sequences = [('WT', TP53_CANONICAL_SEQUENCE)]
    for allele in ALLELES[1:]:
        mut_seq = create_mutant_sequence(allele)
        sequences.append((allele, mut_seq))

    # Calcola embeddings
    esm_embeddings = {}
    with torch.no_grad():
        for name, seq in sequences:
            _, _, tokens = batch_converter([(name, seq)])
            tokens = tokens.to(DEVICE)
            results = esm_model(tokens, repr_layers=[33])
            embedding = results['representations'][33][0, 1:-1].mean(dim=0).cpu()
            esm_embeddings[name] = embedding
            print(f"  {name}: shape={embedding.shape}, norm={embedding.norm():.2f}")

    # Libera memoria GPU
    del esm_model
    torch.cuda.empty_cache()
    print(f"\nEmbeddings ESM-2 reali generati per {len(esm_embeddings)} alleli!")

except Exception as e:
    print(f"ESM-2 non disponibile: {e}")
    print("Uso placeholder embeddings (random)...")
    torch.manual_seed(42)
    esm_embeddings = {}
    for allele in ALLELES:
        esm_embeddings[allele] = torch.randn(ESM_DIM)
    print(f"Placeholder embeddings per {len(esm_embeddings)} alleli")

## 4. Preparazione Dati

In [ ]:
from tsgnn.training.trainer import create_synthetic_training_data
import random
import numpy as np

# Riproducibilita'
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# GRN sintetico (sostituire con GRN reale da pySCENIC/STRING)
edge_index = torch.randint(0, N, (2, E))

# Dati di training sintetici
train_alleles = ['WT', 'R175H', 'R273H', 'R248W']
train_data, val_data, _ = create_synthetic_training_data(
    N=N, E=E, K=K, input_dim=N, alleles=train_alleles,
)

# Usa embeddings ESM-2 reali
train_esm = {a: esm_embeddings[a] for a in train_alleles}

print(f"Train: {len(train_data)} alleli, K={K} bins, N={N} geni")
print(f"Val:   {len(val_data)} alleli")
print(f"GRN:   {E} edges")

## 5. Costruzione Modello

In [ ]:
model = TSGNN(
    num_nodes=N,
    num_edges=E,
    stalk_dim=d,
    input_dim=N,
    esm_dim=ESM_DIM,
    conditioning_dim=128,
    edge_index=edge_index,
    num_diffusion_steps=3,
    use_allele_conditioning=True,
)

params = model.count_parameters()
print(f"Parametri totali: {params['total']:,}")
for name, count in params.items():
    if name != 'total':
        print(f"  {name}: {count:,}")

# Stima memoria
mem_mb = params['total'] * 4 / 1e6  # float32
print(f"\nMemoria modello: ~{mem_mb:.1f} MB")
print(f"Sheaf Laplacian: {N*d}x{N*d} = {(N*d)**2 * 4 / 1e6:.1f} MB per step")

## 6. Training

In [ ]:
%%time

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')

trainer = TSGNNTrainer(
    model=model,
    config=config,
    device=DEVICE,
    checkpoint_dir=f'{PROJECT_DIR}/checkpoints',
    use_wandb=config['training'].get('use_wandb', False),
)

results = trainer.train(
    train_data=train_data,
    val_data=val_data,
    esm_embeddings=train_esm,
)

print(f"\n{'='*50}")
print(f"Training completato!")
print(f"  Epoche: {results['epochs_trained']}")
print(f"  Best val loss: {results['best_val_loss']:.6f}")
print(f"  Tempo totale: {results['total_time']:.0f}s")

In [ ]:
# Plot della learning curve
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

epochs = range(len(results['history']['train_loss']))

ax1.plot(epochs, results['history']['train_loss'], label='Train', linewidth=2)
ax1.plot(epochs, results['history']['val_loss'], label='Validation', linewidth=2)
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Learning Curve')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_yscale('log')

ax2.plot(epochs, results['history']['lr'], color='green', linewidth=2)
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Learning Rate')
ax2.set_title('Learning Rate Schedule')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Evaluation

In [ ]:
from tsgnn.evaluation.metrics import (
    evaluate_all, rewiring_distinguishability,
    extract_differential_edges, compute_hub_influence_scores
)

eval_results = evaluate_all(
    model=model,
    test_data=val_data,
    esm_embeddings=train_esm,
    stalk_dim=d,
)

print("Risultati evaluation:")
print("=" * 60)
for metric, values in eval_results.items():
    print(f"\n{metric}:")
    if isinstance(values, dict):
        for k, v in values.items():
            print(f"  {k}: {v}")

## 8. Analisi Restriction Maps

In [ ]:
from tsgnn.visualization.attention import analyze_regulatory_mode
from tsgnn.visualization.rewiring import identify_bottleneck_timepoints

# Esegui forward pass per ottenere le traiettorie
model.eval()
allele_outputs = {}

with torch.no_grad():
    for allele in train_alleles:
        node_seq = val_data[allele]['node_features_seq'].to(DEVICE)
        allele_emb = train_esm[allele].to(DEVICE)
        preds, maps_traj, laps = model(node_seq, allele_emb)
        allele_outputs[allele] = {
            'predictions': [p.cpu() for p in preds],
            'maps_trajectory': [m.cpu() for m in maps_traj],
            'laplacians': [l.cpu() for l in laps],
        }

# Analisi modalita' regolatorie
for allele in train_alleles:
    maps = allele_outputs[allele]['maps_trajectory']
    modes = analyze_regulatory_mode(maps, edge_index)
    mode_counts = {}
    for v in modes.values():
        mode_counts[v['mode']] = mode_counts.get(v['mode'], 0) + 1
    print(f"{allele}: {mode_counts}")

In [ ]:
# Bottleneck analysis - finestre terapeutiche
for allele in ['R175H', 'R273H']:
    laps = allele_outputs[allele]['laplacians']
    bottlenecks = identify_bottleneck_timepoints(laps, edge_index, d)
    print(f"\n{allele} - Top 3 bottleneck transitions:")
    for bn in bottlenecks[:3]:
        print(f"  t={bn['time_index']}: centrality change={bn['centrality_change']:.4f}")

## 9. Visualizzazioni

In [ ]:
from tsgnn.visualization.rewiring import plot_rewiring_trajectory, plot_differential_rewiring

FIG_DIR = f'{PROJECT_DIR}/figures'
os.makedirs(FIG_DIR, exist_ok=True)

# Rewiring trajectory per allele
for allele in ['R175H', 'R273H']:
    plot_rewiring_trajectory(
        allele_outputs[allele]['laplacians'],
        edge_index, d,
        allele=allele,
        output_dir=FIG_DIR,
    )

# Differential rewiring R175H vs R273H
plot_differential_rewiring(
    allele_outputs['R175H']['laplacians'],
    allele_outputs['R273H']['laplacians'],
    edge_index, d,
    allele_a='R175H', allele_b='R273H',
    output_dir=FIG_DIR,
)
print("Plot salvati!")

In [ ]:
# Mostra i plot inline
from IPython.display import display, Image as IPImage
import glob

for png in sorted(glob.glob(f'{FIG_DIR}/*.png')):
    print(f"\n{os.path.basename(png)}")
    display(IPImage(filename=png, width=700))

In [ ]:
# Allele embedding space
from tsgnn.visualization.networks import plot_allele_embedding_space

plot_allele_embedding_space(
    esm_embeddings,
    method='tsne',
    output_dir=FIG_DIR,
)

display(IPImage(filename=f'{FIG_DIR}/allele_embeddings_tsne.png', width=600))

## 10. Baselines Comparison

In [ ]:
from tsgnn.model.baselines import EvolveGCN, TemporalGAT, create_tsgnn_no_allele

baselines = {
    'TS-GNN (full)': model,
    'EvolveGCN': EvolveGCN(
        num_nodes=N, input_dim=N, hidden_dim=d*4, edge_index=edge_index,
    ),
    'TS-GNN (no allele)': create_tsgnn_no_allele(
        num_nodes=N, num_edges=E, stalk_dim=d, input_dim=N, edge_index=edge_index,
    ),
    'Temporal GAT': TemporalGAT(
        num_nodes=N, input_dim=N, hidden_dim=d*4, edge_index=edge_index,
    ),
}

print("Parametri per modello:")
for name, m in baselines.items():
    n_params = sum(p.numel() for p in m.parameters())
    print(f"  {name}: {n_params:,}")

In [ ]:
from tsgnn.evaluation.benchmarks import BenchmarkRunner

runner = BenchmarkRunner(
    models=baselines,
    test_data=val_data,
    esm_embeddings=train_esm,
    stalk_dim=d,
    seeds=[42],
)

bench_results = runner.run_all()

print("\nRisultati benchmark:")
for model_name, metrics in bench_results.items():
    print(f"\n{model_name}:")
    for metric, val in list(metrics.items())[:5]:
        if isinstance(val, dict) and 'mean' in val:
            print(f"  {metric}: {val['mean']:.4f}")

## 11. Salva risultati su Drive

In [ ]:
import json

# Salva risultati
results_dir = f'{PROJECT_DIR}/results'
os.makedirs(results_dir, exist_ok=True)

# Training history
with open(f'{results_dir}/training_history.json', 'w') as f:
    json.dump(results['history'], f, indent=2)

# Evaluation results
eval_serializable = {}
for k, v in eval_results.items():
    eval_serializable[k] = {kk: float(vv) if isinstance(vv, (int, float, np.floating)) else str(vv)
                            for kk, vv in v.items()} if isinstance(v, dict) else str(v)
with open(f'{results_dir}/evaluation.json', 'w') as f:
    json.dump(eval_serializable, f, indent=2)

# Checkpoint
torch.save(model.state_dict(), f'{PROJECT_DIR}/checkpoints/final_model.pt')

# Copia tutto su Drive
DRIVE_DIR = '/content/drive/MyDrive/ts-gnn-results'
!mkdir -p {DRIVE_DIR}
!cp -r {results_dir}/* {DRIVE_DIR}/
!cp -r {FIG_DIR}/* {DRIVE_DIR}/
!cp {PROJECT_DIR}/checkpoints/final_model.pt {DRIVE_DIR}/

print(f"Tutto salvato in {DRIVE_DIR}")
print(f"  - training_history.json")
print(f"  - evaluation.json")
print(f"  - final_model.pt")
print(f"  - figure .pdf e .png")

## 12. Unit Tests (verifica integrita')

In [ ]:
!cd {PROJECT_DIR} && python -m pytest tests/ -v 2>&1

---
## Note

**Per usare dati reali invece di sintetici:**

1. Scarica GSE178341 (CRC scRNA-seq) da GEO
2. Esegui preprocessing con `tsgnn.data.preprocess.preprocess_scrna()`
3. Costruisci il GRN con `tsgnn.data.grn_construction.construct_base_grn()`
4. Calcola pseudotime con `tsgnn.data.temporal.compute_pseudotime()`
5. Sostituisci `create_synthetic_training_data()` con dati reali

**Per training piu' lungo:**
- Aumenta `max_epochs` a 500
- Usa Colab Pro per sessioni piu' lunghe
- Salva checkpoint periodicamente su Drive

**Hardware:**
- T4 (15 GB VRAM): OK per N=500, d=4
- A100 (40 GB): necessario per N>1000 o d=8